# CSI 4142 - Imputation - Spanish Wine Quality Dataset

| Name          | Student Number    | Group |
|:-------------:|:-----------------:|:-----:|
| Arda Barak    | 300129340         | 90    |



In [1]:
try:
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_absolute_error
    from sklearn.impute import KNNImputer 
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.neighbors import KNeighborsClassifier


except ImportError:
    %pip install pandas
    %pip install seaborn 
    %pip install matplotlib 
    %pip install numpy 

    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_absolute_error
    from sklearn.impute import KNNImputer
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.neighbors import KNeighborsClassifier

sns.set_style("darkgrid")
np.random.seed(42)

## Introduction
Quality is commonly the top desired feature customers look for when buying a wine. The goal of the experimentations produced in this notebook are to apply different data removal methods (MCAR, MAR, MNAR) on the "Spanish Wine Quality" dataset, then use different data imputation methods to evaluate the imputation performance results through the accuracy scores.  

## Dataset Description

Description of the dataset  
|Title  |Description|
|:------|:----------|
|Dataset name                   | Spanish Wine Quality Dataset  |
|Author                         | fedesoriano                   |
|Purpose                        | Created to classify the red wine variants of Spain and to predict the quality of a wine with the related attributes provided | 
|Shape                          | (7500, 11)                    |

Features  
|Attribute      | Description                               |Type                        |
|:--------------|:------------------------------------------|:---------------------------|
|winery:        | Winery name                                                                                                       | Categorical   |
|wine:          | Name of the wine                                                                                                  | Categorical   |
|year:          | Year in which the grapes have been harvested                                                                      | Numerical     |
|rating:        | Average rating given to the wine by users [1 - 5]                                                                 | Numerical     |
|num_reviews:   | Number of users reviewed the wine                                                                                 | Numerical     |
|country:       | Country of origin                                                                                                 | Categorical   |
|region:        | Region of the wine                                                                                                | Categorical   |
|price:         | Price in euros                                                                                                    | Numerical     |
|type:          | Wine variety                                                                                                      | Categorical   |
|body:          | Body score, defined as the richness and weight of the wine in your mouth [1 - 5   ]                               | Numerical     |
|acidity:       | Acidity score, defined as wine's “pucker” or tartness; what makes a wine refreshing and want another sip [1 - 5]  | Numerical     |



In [2]:
# https://raw.githubusercontent.com/ardabarak/4142/refs/heads/main2/wines_SPA.csv
wineDatasetURL =      "https://raw.githubusercontent.com/ardabarak/4142/refs/heads/main2/wines_SPA.csv"

wineDS = pd.read_csv(wineDatasetURL)
print(wineDS.shape)
wineDS.head()

(7500, 11)


,winery,wine,year,rating,num_reviews,country,region,price,type,body,acidity
0,Teso La Monja,Tinto,2013,4.9,58,Espana,Toro,995.00,Toro Red,5.0,3.0
1,Artadi,Vina El Pison,2018,4.9,31,Espana,Vino de Espana,313.50,Tempranillo,4.0,2.0
2,Vega Sicilia,Unico,2009,4.8,1793,Espana,Ribera del Duero,324.95,Ribera Del Duero Red,5.0,3.0
3,Vega Sicilia,Unico,1999,4.8,1705,Espana,Ribera del Duero,692.96,Ribera Del Duero Red,5.0,3.0
4,Vega Sicilia,Unico,1996,4.8,1309,Espana,Ribera del Duero,778.06,Ribera Del Duero Red,5.0,3.0


In [3]:
# checking missing values for each column
print("Missing value of columns: ", wineDS.isnull().sum())

Missing value of columns:  winery            0
wine              0
year              2
rating            0
num_reviews       0
country           0
region            0
price             0
type            545
body           1169
acidity        1169
dtype: int64


In [4]:
# describing the dataset for the numerical variables
wineDS.describe()

,rating,num_reviews,price,body,acidity
count,7500.000000,7500.000000,7500.000000,6331.000000,6331.000000
mean,4.254933,451.109067,60.095822,4.158427,2.946612
std,0.118029,723.001856,150.356676,0.583352,0.248202
min,4.200000,25.000000,4.990000,2.000000,1.000000
25%,4.200000,389.000000,18.900000,4.000000,3.000000
50%,4.200000,404.000000,28.530000,4.000000,3.000000
75%,4.200000,415.000000,51.350000,5.000000,3.000000
max,4.900000,32624.000000,3119.080000,5.000000,3.000000


In [5]:
# dropping the already existing rows with null values to not further mix with the future testings
wineDS.replace('?', pd.NA, inplace=True)
wineDS.replace('', pd.NA, inplace=True)
wineDS.dropna(subset=['year', 'type', 'body', 'acidity'], inplace=True)
print(wineDS.shape)
wineDS.head()

(6329, 11)


,winery,wine,year,rating,num_reviews,country,region,price,type,body,acidity
0,Teso La Monja,Tinto,2013,4.9,58,Espana,Toro,995.00,Toro Red,5.0,3.0
1,Artadi,Vina El Pison,2018,4.9,31,Espana,Vino de Espana,313.50,Tempranillo,4.0,2.0
2,Vega Sicilia,Unico,2009,4.8,1793,Espana,Ribera del Duero,324.95,Ribera Del Duero Red,5.0,3.0
3,Vega Sicilia,Unico,1999,4.8,1705,Espana,Ribera del Duero,692.96,Ribera Del Duero Red,5.0,3.0
4,Vega Sicilia,Unico,1996,4.8,1309,Espana,Ribera del Duero,778.06,Ribera Del Duero Red,5.0,3.0


In [6]:
#dropping the  'country' as there's only 1 unique value in the dataset (Spain)
wineDS = wineDS.drop(columns=['country'])
wineDS.head()

,winery,wine,year,rating,num_reviews,region,price,type,body,acidity
0,Teso La Monja,Tinto,2013,4.9,58,Toro,995.00,Toro Red,5.0,3.0
1,Artadi,Vina El Pison,2018,4.9,31,Vino de Espana,313.50,Tempranillo,4.0,2.0
2,Vega Sicilia,Unico,2009,4.8,1793,Ribera del Duero,324.95,Ribera Del Duero Red,5.0,3.0
3,Vega Sicilia,Unico,1999,4.8,1705,Ribera del Duero,692.96,Ribera Del Duero Red,5.0,3.0
4,Vega Sicilia,Unico,1996,4.8,1309,Ribera del Duero,778.06,Ribera Del Duero Red,5.0,3.0


In [7]:
# checking for missing values again
print("Missing value of columns: ", wineDS.isnull().sum())

Missing value of columns:  winery         0
wine           0
year           0
rating         0
num_reviews    0
region         0
price          0
type           0
body           0
acidity        0
dtype: int64


In [8]:
# applying one hot encoding for the categorical data attributes
wineDS = pd.get_dummies(wineDS, columns=['winery', 'wine','region','type'], drop_first=True)

wineDS.head()

,year,rating,num_reviews,price,body,acidity,winery_Aalto,winery_Abadal,winery_Abadia Retuerta,winery_Abel Mendoza Monge,...,type_Ribera Del Duero Red,type_Rioja Red,type_Rioja White,type_Sauvignon Blanc,type_Sherry,type_Sparkling,type_Syrah,type_Tempranillo,type_Toro Red,type_Verdejo
0,2013,4.9,58,995.00,5.0,3.0,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
1,2018,4.9,31,313.50,4.0,2.0,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,2009,4.8,1793,324.95,5.0,3.0,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
3,1999,4.8,1705,692.96,5.0,3.0,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,1996,4.8,1309,778.06,5.0,3.0,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False


## Experiments




### Experiment 1
- a) Attribute to be imputed        :   rating  
- b) Simulated type of data missing :   MCAR  
- c) Imputation type applied        :   Correlational imputation  
- d) Effectivity of the approach    :   Please see results below  



In [9]:
# EXPERIMENT 1
copyDS1 = wineDS.copy()                             # deep copying the dataset for the experiment
copyDS1['year'] = pd.to_numeric(copyDS1['year'], errors='coerce')  # converting 'year' to numeric

# randomly removing 1/10 of the 'rating' values
missingRating = np.random.choice(copyDS1.dropna(subset=['rating']).index, size=int(0.1 * len(copyDS1)), replace=False)
trueValsRating = copyDS1.loc[missingRating, 'rating'].copy()    # storing the removed values for evaluation
copyDS1.loc[missingRating, 'rating'] = np.nan                   # missing values
correlationMatrix = copyDS1.corr()                              # checking for most related features
top_features = correlationMatrix['rating'].abs().sort_values(ascending=False).index[1:4]

trainData = copyDS1.dropna(subset=['rating'])       # training the model
xTrain = trainData[top_features].dropna()
yTrain = trainData.loc[xTrain.index, 'rating']      # ytrain matching xtrain 
lregModl = LinearRegression()
lregModl.fit(xTrain, yTrain)
xMissingVals = copyDS1.loc[missingRating, top_features].dropna()    # predicting the missing 'rating' values
if not xMissingVals.empty:                                          # considering only the cases with values
    copyDS1.loc[xMissingVals.index, 'rating'] = lregModl.predict(xMissingVals)

imputedValsRating = copyDS1.loc[missingRating, 'rating'].dropna()   # evaluating from the Mean Absolute Error values
trueValsRating = trueValsRating.loc[imputedValsRating.index]        #alligning the indexes
meanAbsoulutErr = mean_absolute_error(trueValsRating, imputedValsRating)
print(f"MAE score of Imputation on 'rating': {meanAbsoulutErr:.4f}")


MAE score of Imputation on 'rating': 0.0729


### Analysis of Experiment 1  
- 1st run: 0.0728  
- 2nd run: 0.0649  
- 3rd run: 0.0722  
- As these scores are on average deviating from the imputed value by on average "0.0699" to the original value in a complete scale of 0-5, it shows us a low error and suggests the imputation method is effective.

### Experiment 2
- a) Attribute to be imputed        : price  
- b) Simulated type of data missing : MNAR  
- c) Imputation type applied        : Regression imputation  
- d) Effectivity of the approach    : Please see results below  


In [10]:
# EXPERIMENT 2
copyDS2 = wineDS.copy()
copyDS2['year'] = pd.to_numeric(copyDS1['year'], errors='coerce')

# knowing MNAR is in a way related with the data provided(or knowingly avoided) I assume the higher price items make more sense to be missing
mnarThreshold = copyDS2['price'].quantile(0.75)             # wines above the 75th percentile price as the threshold
missingPrice = copyDS2[copyDS2['price'] > mnarThreshold].sample(frac=0.5, random_state=42).index  # dropping randomly half of them
trueValsPrice = copyDS2.loc[missingPrice, 'price'].copy()   # keeping original values for evaluation
copyDS2.loc[missingPrice, 'price'] = np.nan                 # adding the missing values

features = copyDS2.drop(columns=['price']).select_dtypes(include=['number']).columns  # selecting the features to train the model with
trainData = copyDS2.dropna(subset=['price'])    # splitting data to training & missing
xTrain = trainData[features].dropna(axis=1)     # dropping columns with missing values
yTrain = trainData['price']

regModel = LinearRegression()   # training the regressional model
regModel.fit(xTrain, yTrain)
xMissingVals = copyDS2.loc[missingPrice, features].dropna(axis=1)   # predicting missing values
if not xMissingVals.empty:
    copyDS2.loc[xMissingVals.index, 'price'] = regModel.predict(xMissingVals)

imputedValsPrice = copyDS2.loc[missingPrice, 'price'].dropna()
trueValsPrice = trueValsPrice.loc[imputedValsPrice.index]
meanAbsErr = mean_absolute_error(trueValsPrice, imputedValsPrice)
print(f"MAE score of Imputation of 'price': {meanAbsErr:.4f}")


MAE score of Imputation of 'price': 109.6698


### Analysis of Experiment 2  
- result: 109.6698
- The MAE score of 109.66 means the imputed prices differ by ~110 euros, which is a high error rate. Considering the results being off this high of the actual prices, the result suggests is not well fit to use regression imputation on data where it is hard to make linear relations.
- Unlike the previous (MCAR) data missing format, as this one is missing Not at random, I recorded the result once as it will give the same result each time.


### Experiment 3
- a) Attribute to be imputed        : region  
- b) Simulated type of data missing : MAR  
- c) Imputation type applied        : Similarity based imputation  
- d) Effectivity of the approach    : Please see the results below  


In [11]:
# EXPERIMENT 3
#copied the previous(above) code for easier access for the experiment
yedekWDS = pd.read_csv(wineDatasetURL)
yedekWDS.replace('?', pd.NA, inplace=True)
yedekWDS.replace('', pd.NA, inplace=True)
yedekWDS.dropna(subset=['year', 'type', 'body', 'acidity'], inplace=True)
yedekWDS = yedekWDS.drop(columns=['country'])
copyDS3 = yedekWDS.copy()
copyDS3['year'] = pd.to_numeric(copyDS3['year'], errors='coerce')

lowRatingThreshold = copyDS3['rating'].quantile(0.25)   # wines in the lowest 1/4 rating
copyDS3.reset_index(drop=True, inplace=True)            # resetting index
missingRegion = copyDS3[copyDS3['rating'] <= lowRatingThreshold].sample(frac=0.5, random_state=42).index  
trueValsRegion = copyDS3.loc[missingRegion, 'region'].copy()    # storing original values
copyDS3.loc[missingRegion, 'region'] = np.nan                   # creating missing values
categorical_features = ['winery', 'wine', 'region', 'type']     # One Hot on categorical features
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_categorical = encoder.fit_transform(copyDS3[categorical_features])

# converting the dataset and merging with the originals
encoded_categorical_df = pd.DataFrame(encoded_categorical, columns=encoder.get_feature_names_out(categorical_features))
copyDS3 = pd.concat([copyDS3.reset_index(drop=True), encoded_categorical_df.reset_index(drop=True)], axis=1)

# picking numerical features for imputations
features = copyDS3.drop(columns=['region'] + categorical_features).select_dtypes(include=['number']).columns  
knnImputer = KNNImputer(n_neighbors=5, weights='uniform')
imputedData = knnImputer.fit_transform(copyDS3[features])
copyDS3[features] = imputedData                             # updating imputed values
knnClassifier = KNeighborsClassifier(n_neighbors=15)        # training on values
trainData = copyDS3.dropna(subset=['region'])
knnClassifier.fit(trainData[features], trainData['region'])
missingFeatures = copyDS3.loc[missingRegion, features]      # predicting the missing values
predictedRegions = knnClassifier.predict(missingFeatures)
copyDS3.loc[missingRegion, 'region'] = predictedRegions     # filling missing vals

imputedValsRegion = copyDS3.loc[missingRegion, 'region'].dropna()
trueValsRegion = trueValsRegion.loc[imputedValsRegion.index]
accuracy = (imputedValsRegion == trueValsRegion).mean() * 100
print(f"Accuracy score of knn imputation for 'region': {accuracy:.2f}%")


Accuracy score of knn imputation for 'region': 97.60%


### Analysis of Experiment 3  
Results from 3 different runs:
| neighbour size set | accuracy score |
|:---:|:---:|
|5 |97.39%  |
|10|97.43%  |
|15|97.60%  |

- The results show that the knn imputation was highly accurate in imputing the missing region values. The slight increase in the accuracy scores in the results as we increased the neighbours show with more neighbours, the model gains slightly improved accuracy scores.  


## Conclusion

In conclusion, in this notebook we explored 3 different missing data types as MCAR, MAR, and MNAR applied to the Spanish Wine Quality dataset. Then evaluated the effectivity of the Correlational, Regression, and Knn imputation methods.  
The results of these 3 experiments showed that the imputation choice we apply effects the imputation accuracy scores and should be aligning with a matching data missing type, as a good match can give promising imputated results as high as ~97% while an unfit match may be quite lower and ineffective.  

## References
- https://www.kaggle.com/datasets/fedesoriano/spanish-wine-quality-dataset/data  
- https://saturncloud.io/blog/how-to-delete-rows-with-null-values-in-a-specific-column-in-pandas-dataframe/  
- https://www.digitalocean.com/community/tutorials/pandas-dropna-drop-null-na-values-from-dataframe  
- https://stackoverflow.com/questions/78647587/how-can-you-generate-mcar-mnar-and-mar-missingness-pattern-in-a-dataset-using-p  
- https://www.kaggle.com/code/yassirarezki/handling-missing-data-mcar-mar-and-mnar-part-i  
- https://docs.python.org/3/library/copy.html  
- https://stackoverflow.com/questions/69061768/introduce-missingness-into-mixed-data-using-mar-mnar-and-mcar-in-python  
- https://rmisstastic.netlify.app/how-to/python/generate_html/how%20to%20generate%20missing%20values  
